In [1]:
import numpy as np
import Wavelet2D1T as wt
import scipy
import matplotlib.pyplot as plt
from itertools import product
from tqdm.notebook import tqdm
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata
%matplotlib notebook

将参数空间分块处理，计算小波功率谱

In [2]:
def split_array(m, n):
    q, r = divmod(m, n)
    index = []
    start = 0

    for i in range(n):
        length = q + 1 if i < r else q
        index.append((start, start + length))
        start += length

    return index

def get_omega_kr(data, kr_array, omega_array, vel_arr, rot_arr, dx, dy, dt, blocks=(4, 4, 1, 1)):
    # 初始化结果数组
    p = np.zeros((len(kr_array), len(omega_array), len(vel_arr), len(rot_arr)))
    fft_data = scipy.fft.fftn(data)

    # 划分参数数组的索引范围（每个元素为(start, end)）
    sub_kr_index = split_array(len(kr_array), blocks[0])  # kr的块索引
    sub_omega_index = split_array(len(omega_array), blocks[1])  # omega的块索引
    sub_vel_index = split_array(len(vel_arr), blocks[2])  # vel的块索引
    sub_rot_index = split_array(len(rot_arr), blocks[3])  # rot_angle的索引

    # 生成三个维度块索引的所有组合（笛卡尔积），用单重循环遍历
    total = len(sub_kr_index) * len(sub_omega_index) * len(sub_vel_index) * len(sub_rot_index)
    for kr_idx, omega_idx, vel_idx, rot_idx in tqdm(product(sub_kr_index, sub_omega_index, sub_vel_index, sub_rot_index),
                                           total=total, desc='processing parameter blocks'):
        # 提取当前块的子数组
        sub_kr = kr_array[kr_idx[0]:kr_idx[1]]
        sub_omega = omega_array[omega_idx[0]:omega_idx[1]]
        sub_vel = vel_arr[vel_idx[0]:vel_idx[1]]
        sub_rot = rot_arr[rot_idx[0]:rot_idx[1]]

        # 计算小波变换
        w = wt.cwt2DT(fft_data, 1/dx, 1/dy, 1/dt, 2*np.pi/sub_kr, 2*np.pi/sub_omega, sub_vel, sub_rot)

        # 计算功率谱（积分x y t）
        p_kr_omega = np.sum(np.abs(w)**2, axis=(0, 1, 2))

        # 将结果写入p的对应位置
        p[kr_idx[0]:kr_idx[1], omega_idx[0]:omega_idx[1], vel_idx[0]:vel_idx[1], rot_idx[0]:rot_idx[1]] = p_kr_omega
    # 对x、y、t求平均
    p/=(data.shape[0]*data.shape[1]*data.shape[2])
    if len(rot_arr) == 1:
        return p[:, :, :, 0]

    p = p.transpose(0, 3, 1, 2) # 维度为(k, theta, omega, c_vel)
    p_flat = p.reshape(-1, p.shape[2], p.shape[3])

    # 转换为kx，ky坐标
    rot_arr_rad = rot_arr/180*np.pi
    k_grid, theta_grid = np.meshgrid(kr_array, rot_arr_rad, indexing='ij')
    kx = (k_grid * np.cos(theta_grid)).flatten()
    ky = (k_grid * np.sin(theta_grid)).flatten()

    points = np.column_stack((kx, ky))
    kx_array = np.linspace(kx.min()/np.sqrt(2), kx.max()/np.sqrt(2), 500)
    ky_array = np.linspace(ky.min()/np.sqrt(2), ky.max()/np.sqrt(2), 500)
    kx_i_grid, ky_i_grid = np.meshgrid(kx_array, ky_array, indexing='ij')

    # 插值
    p_flat_reshaped = p_flat.reshape(p_flat.shape[0], -1)
    p_xy_flat = griddata(
        points=points,
        values=p_flat_reshaped,
        xi=(kx_i_grid, ky_i_grid),
        method='linear'
    )
    p_xy_flat = np.nan_to_num(p_xy_flat, nan=0)
    p_xy = p_xy_flat.reshape(kx_i_grid.shape[0], ky_i_grid.shape[1], p.shape[2], p.shape[3])

    return kx_array, p, np.trapezoid(y=p_xy, x=ky_array, axis=1)


### 对生成的信号进行测试：
$$
f\left(x, y, t\right) = sin\left(3.3x - 1.1t\right) + sin\left(6.7x - 3.3t\right)
$$

In [19]:
nx = 40
ny = 50
nt = 60
x = np.arange(nx)
y = np.arange(ny)
t = np.arange(nt)
X, Y, T = np.meshgrid(x, y, t, indexing='ij')
f_vals = 10*np.cos(0.6*X + 0.8*Y - T) + 10*np.sin(1.9*X - 1.7*T) + 10*np.cos(1*X + 2.4*Y - 2.7*T)

kr_array = np.linspace(0.1, 6, 50)
omega_array = np.linspace(0.1, 6, 50)
rot_array = np.linspace(0, 179, 10)

kx_array, p, p_i = get_omega_kr(f_vals, kr_array, omega_array, [1], rot_array, x[1]-x[0], y[1]-y[0], t[1]-t[0], (32, 32, 1, 10))

processing parameter blocks:   0%|          | 0/10240 [00:00<?, ?it/s]

In [20]:
from matplotlib.animation import FuncAnimation
fig, ax = plt.subplots(figsize=(4, 5))
im = ax.imshow(f_vals[:, :, 0], origin='lower', cmap='coolwarm', vmin=np.min(f_vals), vmax=np.max(f_vals))
ax.set_xlabel('y')
ax.set_ylabel('x')
ax.set_title('simulate signal')
plt.colorbar(im, ax=ax)
def update(frame):
    im.set_data(f_vals[:, :, frame])
    return im
ani = FuncAnimation(fig=fig, func=update, frames=len(t), interval=25)
# ani.save('../fig/模拟测试信号.gif', fps=80, dpi=200)

<IPython.core.display.Javascript object>

In [23]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(p_i[:, :, 0], origin='lower', cmap='coolwarm', extent=(omega_array[0], omega_array[-1], kx_array[0], kx_array[-1]))
ax.set_aspect('auto')
ax.set_xlabel('omega')
ax.set_ylabel('kx')
plt.colorbar(im, ax=ax)
fig.show()
# fig.savefig('../fig/迷你信号测试结果.png', dpi=200)

<IPython.core.display.Javascript object>

### 对UCoMP图像进行测试

In [ ]:
kr_array = np.linspace(-0.15, 0.15, 100)
omega_array = np.linspace(0.001, 0.18, 100)
ucomp_test = np.load('../data/polar_datas_20221122.npy').transpose(1, 2, 0)
# fig, ax = plt.subplots(figsize=(6, 2))
# ax.imshow(ucomp_test[:, :, 0], origin='lower', cmap='coolwarm')

In [ ]:
# UCoMP仪器分辨率为2.944arcsec，对应大约1AU*6/3600/180*pi=2.1691Mm，时间分辨率约为33.65s
p = get_omega_kr(ucomp_test, kr_array, omega_array, [1], 2.1691, 2.1691, 33.65, (32, 32, 1))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(p[:, :, 0], origin='lower', cmap='coolwarm', extent=(omega_array[0], omega_array[-1], kr_array[0], kr_array[-1]), norm=LogNorm())
ax.set_aspect('auto')
ax.set_xlabel(r'$\omega/s^{-1}$')
ax.set_ylabel(r'$k_r/Mm^{-1}$')
plt.colorbar(im, ax=ax)
fig.show()
# fig.savefig('../fig/log极坐标数据三维小波变换结果20220205.png', dpi=300)

In [ ]:
kr_array = np.linspace(0.2, 0.7, 25)
omega_array = np.linspace(0.1, 3, 25)
ucomp_test = np.load('./data/cut_simulation_xy.npy').transpose(1, 2, 0)

In [ ]:
# 分辨率0.003Rs~2.1Mm
p = get_omega_kr(ucomp_test, kr_array, omega_array, [1], 2.1, 2.1, 1, (32, 32, 1))

### morton数据测试

In [25]:
import os
comp_fits_dir = "E:/python/projects/UCoMP数据处理/data/fits_20120327_morton/"
# comp_fits_dir = '../mnt/ucomp/'
comp_fits_list = os.listdir(comp_fits_dir)
already_crop_data = False

for i, comp_fits in enumerate(comp_fits_list):
    from astropy.io import fits
    hdul = fits.open(comp_fits_dir+comp_fits)
    obstime = hdul[0].header['DATE-OBS']+' '+hdul[0].header['TIME-OBS']
    if i == 0:
        print(hdul[0].header['CDELT1'])
    doppler_map = hdul[3].data
    if not already_crop_data:
        sequence_doppler_map = np.copy(doppler_map[530:590, 250:370])
        already_crop_data = True
    else:
        sequence_doppler_map = np.dstack([sequence_doppler_map, doppler_map[530:590, 250:370]])

input_data_3D = np.transpose(sequence_doppler_map, (0,1,2))[:, :, 35:170]

4.46


In [ ]:
from matplotlib.animation import FuncAnimation
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(input_data_3D[:, :, 0], origin='lower', cmap='coolwarm', vmin=-10, vmax=10)
ax.set_xlabel('theta')
ax.set_ylabel('r')
ax.set_aspect('equal')
plt.colorbar(im, ax=ax)
def update(frame):
    im.set_data(input_data_3D[:, :, frame])
    return im
ani = FuncAnimation(fig=fig, func=update, frames=input_data_3D.shape[2], interval=25)

In [ ]:
ani.save('./fig/comp_data_morton.gif', fps=20)

In [5]:
kr_array = np.logspace(-2, -0.8, 100)
omega_array = np.logspace(-4, -0.6, 100)
rot_array = np.linspace(0, 179, 10)
# kr_array = 2*np.pi/np.arange(60, 1, -1)/3.286
# omega_array = 2*np.pi/np.arange(135, 1, -1)/30
kx_array, p, p_i = get_omega_kr(input_data_3D, kr_array, omega_array, [1], rot_array, 3.286, 3.286, 30, (64, 64, 1, 10))

In [3]:
# np.save('./data/p_i.npy', p_i)
# np.save('./data/p.npy', p)
# np.save('./data/kx_array.npy', kx_array)
p = np.load('./data/p.npy')
p_i = np.load('./data/p_i.npy')
kx_array = np.load('./data/kx_array.npy')

In [15]:
fig, ax = plt.subplots(figsize=(6, 5))
mesh = ax.pcolormesh(kx_array, omega_array, p_i[:, :, 0].T, shading='nearest', cmap='coolwarm')
ax.set_aspect('auto')
ax.set_ylabel(r'$\omega/s^{-1}$')
ax.set_xlabel(r'$k_r/Mm^{-1}$')
plt.colorbar(mesh, ax=ax)
# fig.show()
# fig.savefig('./fig/wt2DT_morton_linear.png', dpi=300)

<IPython.core.display.Javascript object>